# EGS 2403 Remote Sensing Applications — Week 0: the ideas you need before Week 1

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanibenson/jkuat-geospatial-teaching/blob/main/egs2403/week0-prerequisites.ipynb)

**Self-study, ungraded, about three hours.** Do it before the first lecture.

The unit runs in Google Earth Engine. This notebook deliberately does **not**. Here you
work in plain Python and NumPy on small made-up images, for three reasons:

* It runs offline, on your own laptop or in Colab, before your Earth Engine account exists.
* When you can *see* the array, you can see what an operation does. Earth Engine hides
  the pixels, so it is a hard place to learn what the pixels are doing.
* Everything here carries over to any platform. The last section maps each operation to
  its Earth Engine name, and Appendix D of the reader runs the same analysis without Google.

Each section is built the same way:

1. **The idea**, in plain language.
2. **A small demonstration** on synthetic data where we know the right answer.
3. **Check yourself**: a short exercise. Its check cell says ✓, ✗ with a hint, or ○ not
   attempted yet. A model answer is folded away underneath. Try first, then look.
4. **Where this goes**: the chapter of the reader that leans on it.

### Skip what you already know

If you are confident about a question, skim that section. If you hesitate, do the section properly.

1. Why does healthy vegetation look dark in red and bright in near-infrared?
2. A Sentinel-2 L2A pixel stores `2345`. What is the reflectance? (Two possible answers, depending on the date. Why?)
3. Why does NDVI barely change between a sunlit and a shaded slope, but change a lot under haze?
4. How do you test whether bit 3 of a QA value is set?
5. Why is a median composite better than a mean composite in cloudy months?
6. What is the Vegetation Condition Index, in one formula?
7. What is the difference between user's and producer's accuracy?
8. What is −20 dB as a plain number, and why must you never average in dB?

<details><summary>Short answers</summary>

1. Chlorophyll absorbs red light for photosynthesis, while the internal structure of the
   leaf scatters near-infrared strongly. (§1)
2. `0.2345` if the scene was processed before 25 Jan 2022. After that, a −1000 offset
   applies, giving `0.1345`. Earth Engine's `S2_SR_HARMONIZED` removes the offset for
   you. Always check the scale factors. (§2)
3. Shading *multiplies* every band by the same factor, which cancels in a ratio. Haze
   *adds* to every band, which does not cancel. (§3)
4. `(qa & (1 << 3)) != 0`, or `(qa >> 3) & 1`. (§4)
5. Clouds are extreme values, and the median ignores extremes. The mean is dragged
   towards them. (§5)
6. $\text{VCI} = 100\,(\text{NDVI} - \text{NDVI}_\min)/(\text{NDVI}_\max - \text{NDVI}_\min)$,
   with min and max taken over the same period in past years. (§6)
7. User's: of the pixels *mapped* as class k, the fraction that really are k (the map
   user's view). Producer's: of the pixels that really *are* k, the fraction mapped as k
   (the map maker's view). (§7)
8. $10^{-20/10} = 0.01$. Averaging logarithms is not the same as averaging the values,
   and for speckled radar it biases the result by about −2.5 dB. (§8)
</details>

In [ ]:
# Setup: run this cell first, every time you open the notebook.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.interpolate import PchipInterpolator

plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False})


def check(label, value, test, hint=""):
    # ✓ / ✗ / ○ feedback for the exercises. It never raises, so 'Run all' is safe.
    if value is ...:
        print(f"○ {label}: not attempted yet")
        return
    try:
        ok = bool(test(value))
    except Exception as e:
        ok, hint = False, f"{hint}  (checking raised {type(e).__name__}: {e})"
    print(("✓ " if ok else "✗ ") + label + ("" if ok else f"\n   Hint: {hint}"))


# ---- Stylised reflectance curves (shapes are realistic; values are not measurements)
_anchors = {
    "vegetation": ([400, 450, 550, 670, 700, 760, 900, 1100, 1450, 1650, 1950, 2200, 2500],
                   [.03, .03, .08, .035, .10, .40, .42, .38, .12, .22, .05, .10, .05]),
    "soil":       ([400, 700, 1000, 1400, 1650, 1900, 2200, 2500],
                   [.07, .20, .26, .28, .33, .28, .28, .25]),
    "water":      ([400, 500, 600, 700, 800, 1000, 2500],
                   [.07, .06, .04, .02, .01, .003, .001]),
    "urban":      ([400, 700, 1000, 1650, 2200, 2500],
                   [.09, .14, .19, .24, .22, .20]),
}
curves = {k: PchipInterpolator(*v) for k, v in _anchors.items()}

# Sentinel-2 MSI bands used in this notebook, with central wavelength in nm
BANDS = {"B2": 490, "B3": 560, "B4": 665, "B8": 842, "B11": 1610, "B12": 2190}
CLASSES = ["water", "vegetation", "soil", "urban"]
library = pd.DataFrame({c: [float(curves[c](w)) for w in BANDS.values()] for c in CLASSES},
                       index=list(BANDS)).T                       # rows: classes, cols: bands


def make_scene(seed=0, size=100):
    # A 1 km x 1 km synthetic Sentinel-2 scene at 10 m: a lake, fields, bare soil, a town.
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size]
    cls = np.full((size, size), 2)                                          # soil
    cls[ndimage.gaussian_filter(rng.standard_normal((size, size)), 6) > 0.02] = 1   # vegetation
    cls[(xx > 65) & (yy > 60)] = 3                                          # town
    cls[((xx - 30) / 22) ** 2 + ((yy - 35) / 14) ** 2 < 1] = 0              # lake
    refl = np.stack([library[b].values[cls] for b in BANDS])               # (6, size, size)
    refl = refl * rng.normal(1, 0.06, (1, size, size)) + rng.normal(0, 0.006, refl.shape)
    return cls, np.clip(refl, 0.0005, 1)

print("Ready. Spectral library (reflectance):")
print(library.round(3))

---
## 1 · Light, reflectance and spectral signatures

### The idea

Optical sensors measure sunlight reflected by the surface, in **bands**: windows of
wavelength. The ones you will use:

| Region | Wavelength | Sentinel-2 bands | What it sees |
|---|---|---|---|
| Visible (blue, green, red) | 400–700 nm | B2, B3, B4 | colour; chlorophyll absorbs blue and red |
| Near-infrared (NIR) | 700–1300 nm | B8 (and red-edge B5–B7, B8A) | leaf structure: healthy leaves scatter NIR strongly |
| Shortwave infrared (SWIR) | 1300–2500 nm | B11, B12 | water content of leaves and soil; minerals |
| Thermal infrared | 8–14 µm | none (Landsat B10) | emitted heat, not reflected light (Week 9) |
| Microwave | cm | none (Sentinel-1 radar) | surface roughness and moisture, through cloud (Week 8) |

**Reflectance** is the *fraction* of incoming light that is reflected: a number from 0 to 1,
with no units. Because it is a ratio, it can be compared across dates, sensors and sun angles.
A raw brightness value cannot.

Each material has a characteristic shape of reflectance against wavelength: its
**spectral signature**. Every index, classifier and threshold in this unit exploits
differences between signatures.

### Demonstration

In [ ]:
wl = np.linspace(400, 2500, 500)
colors = {"water": "#2b6cb0", "vegetation": "#2f855a", "soil": "#b7791f", "urban": "#718096"}
fig, ax = plt.subplots(figsize=(9, 3.8))
for c in CLASSES:
    ax.plot(wl, curves[c](wl), color=colors[c], lw=2, label=c)
for b, w in BANDS.items():
    ax.axvline(w, color="k", lw=0.6, ls=":"); ax.text(w, 0.46, b, ha="center", fontsize=8)
ax.axvspan(700, 760, color="#B8443A", alpha=0.08); ax.text(730, 0.02, "red edge", ha="center", fontsize=8)
ax.set(xlabel="wavelength (nm)", ylabel="reflectance", ylim=(0, 0.48),
       title="Stylised spectral signatures, with Sentinel-2 band centres")
ax.legend(loc="upper right", bbox_to_anchor=(1, 0.93)); plt.show()

Look for the **red edge**, vegetation's jump from under 0.05 in red to about 0.4 in NIR
over only 90 nm. No other common surface does this. Look also at **water**, which is
nearly black beyond 800 nm. And **soil**, which rises smoothly with no sharp features.

### Check yourself

**1a.** An unlabelled pixel, `mystery`, has the six band values below. Find the class in
`library` that is **closest** to it (Euclidean distance across the six bands), and store
its name in `mystery_class`. This is a *minimum-distance classifier*, the simplest
classifier there is.

**1b.** Using `library`, which **single band** differs most between vegetation and water?
Store its name (e.g. `"B4"`) in `best_band`.

In [ ]:
mystery = np.array([0.08, 0.13, 0.17, 0.24, 0.31, 0.27])

In [ ]:
mystery_class = ...
best_band = ...

<details><summary>Show a model answer</summary>

```python
dist = np.sqrt(((library.values - mystery) ** 2).sum(axis=1))   # one distance per class
mystery_class = library.index[dist.argmin()]

diff = (library.loc["vegetation"] - library.loc["water"]).abs()
best_band = diff.idxmax()
print(dict(zip(library.index, dist.round(3))), mystery_class, best_band)
```

</details>

In [ ]:
check("1a minimum-distance class", mystery_class, lambda v: v == "soil",
      "Subtract mystery from each row of library.values, square, sum across bands, sqrt, argmin.")
check("1b most separating band", best_band, lambda v: v == "B8",
      "Absolute difference of the two rows, then .idxmax().")

> **Where this goes:** Week 1 *Spectral signatures: the foundation of everything that
> follows*; Appendix B band table. Week 5's classifiers are more sophisticated
> versions of 1a.

---
## 2 · An image is an array of numbers

### The idea

A multispectral image is a 3-D array: **bands × rows × columns**. One pixel is one
column through the stack. Several facts follow that you need every week:

* **Pixel size gives area.** A Sentinel-2 visible or NIR pixel is 10 m × 10 m = 100 m²,
  so 100 pixels make 1 hectare. SWIR bands are 20 m. Landsat is 30 m (900 m²).
* **Storage is integer, reflectance is not.** Files store reflectance as integers to save
  space, and you must apply a **scale factor** and sometimes an **offset**:

| Product | Physical value from stored number |
|---|---|
| Sentinel-2 L2A, before 25 Jan 2022 | reflectance = DN / 10 000 |
| Sentinel-2 L2A, from 25 Jan 2022 (processing baseline 04.00) | reflectance = (DN − 1000) / 10 000 |
| Earth Engine `COPERNICUS/S2_SR_HARMONIZED` | reflectance = DN / 10 000 (the offset is already removed) |
| Landsat Collection 2 Level-2 reflectance | reflectance = DN × 0.0000275 − 0.2 |
| Landsat Collection 2 Level-2 surface temperature | kelvin = DN × 0.00341802 + 149.0 |

* **No-data** pixels (outside the swath, for example) have a fill value, often 0. Turn
  them into `NaN` *before* any maths, or they become fake dark pixels.
* **Integer arithmetic bites.** In unsigned 16-bit integers (`uint16`) there are no
  negative numbers, so `NIR − red` over water, where red > NIR, **wraps round** to
  about 65 000. Convert to float first.

### Demonstration

In [ ]:
cls, refl = make_scene(seed=2)
dn = np.round(refl * 10000 + 1000).astype(np.uint16)       # a post-2022 L2A file
dn[:, :, :8] = 0                                            # a no-data strip at the swath edge
print("array shape (bands, rows, cols):", dn.shape, " dtype:", dn.dtype)

red, nir = dn[2], dn[3]                                     # whole bands, still uint16
r, c = 35, 30                                               # a pixel in the lake
print(f"lake pixel: red DN {red[r, c]}, NIR DN {nir[r, c]}")
print(f"  NIR - red in uint16 : {(nir - red)[r, c]}      <- wrapped round, and no warning")
print(f"  NIR - red in float  : {(nir.astype(float) - red)[r, c]:.0f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
rgb = np.clip(np.stack([refl[2], refl[1], refl[0]], -1) * 3.5, 0, 1)
ax[0].imshow(rgb); ax[0].set_title("true colour (B4, B3, B2)")
ax[1].imshow(cls, cmap=plt.matplotlib.colors.ListedColormap(list(colors.values())))
ax[1].set_title("the classes we built it from")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()

### Check yourself

**2a.** Convert `dn` to a float reflectance array `refl2` with the post-2022 formula, with
`NaN` wherever the stored value is 0.

**2b.** Using `cls` (0 = water), how many **hectares** of lake are in the scene? Store it as `lake_ha`.

In [ ]:
refl2 = ...
lake_ha = ...

<details><summary>Show a model answer</summary>

```python
refl2 = (dn.astype(float) - 1000) / 10000
refl2[dn == 0] = np.nan
lake_ha = np.sum(cls == 0) * 10 * 10 / 10_000       # pixels x m² per pixel / m² per ha
print(np.nanmean(refl2), lake_ha)
```

</details>

In [ ]:
check("2a scaled reflectance", refl2,
      lambda v: (np.isnan(v[:, :, :8]).all() and not np.isnan(v[:, :, 8:]).any()
                 and np.allclose(v[:, :, 8:], refl[:, :, 8:], atol=1e-4)),
      "Cast to float BEFORE subtracting 1000; then set refl2[dn == 0] = np.nan.")
check("2b lake area", lake_ha, lambda v: np.isclose(v, np.sum(cls == 0) / 100),
      "Count water pixels; each is 10 m x 10 m = 100 m²; 1 ha = 10 000 m².")

> **Where this goes:** Week 1 *Processing levels*; Week 3 *What is already done, and
> what is not*; Appendix B *Scale factors: check these every time*. The reader means it.

---
## 3 · Band arithmetic and normalised differences

### The idea

An **index** combines bands to exaggerate one property of the surface. Most of the
common ones are **normalised differences**:

$$ \text{ND}(a, b) = \frac{a - b}{a + b} \qquad \text{always between } -1 \text{ and } +1 $$

* $\text{NDVI} = \text{ND}(\text{NIR}, \text{red})$: green vegetation. It uses the red edge from §1.
* $\text{NDWI} = \text{ND}(\text{green}, \text{NIR})$ (McFeeters): open water.
* The reader's Appendix B lists the rest in Sentinel-2 band names.

**Why normalise?** Suppose a slope faces away from the sun and receives only 45% of the
light. *Every* band is multiplied by 0.45. That factor appears in the top and bottom of the
fraction and cancels. So the index describes the *surface*, not the illumination.

**The limit:** haze and thin cloud **add** light to every band instead of multiplying it,
and an added constant does not cancel. This is one reason atmospheric correction matters.

Also watch out for **division by zero**: where $a + b = 0$ (no-data, or deep shadow) you
need `NaN`, not a crash or an `inf`.

### Demonstration

In [ ]:
veg = library.loc["vegetation"]
def ndvi(p): return (p["B8"] - p["B4"]) / (p["B8"] + p["B4"])

sunlit, shaded, hazy = veg, veg * 0.45, veg + 0.03
for name, p in [("sunlit", sunlit), ("shaded (x0.45)", shaded), ("hazy (+0.03)", hazy)]:
    print(f"{name:15s} red {p['B4']:.3f}  NIR {p['B8']:.3f}   NDVI {ndvi(p):.3f}")

### Check yourself

**3a.** Write `norm_diff(a, b)` that works on whole arrays and returns `NaN`, with no
warning, wherever `a + b == 0`. Hint: `with np.errstate(divide="ignore", invalid="ignore"):`.

**3b.** Compute NDWI for the scene from §2 (use `refl`: B3 is index 1, B8 is index 3),
make a water mask `water = ndwi > 0`, and see how well it matches the true lake. The check
uses **intersection over union** (IoU): overlap divided by combined area. It should be above 0.9.

In [ ]:
def norm_diff(a, b):
    # YOUR CODE HERE
    return ...

water = ...

<details><summary>Show a model answer</summary>

```python
def norm_diff(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        out = (a - b) / (a + b)
    out[~np.isfinite(out)] = np.nan
    return out

ndwi = norm_diff(refl[1], refl[3])
water = ndwi > 0
iou = (water & (cls == 0)).sum() / (water | (cls == 0)).sum()
print(f"IoU with the true lake: {iou:.3f}")
```

</details>

In [ ]:
check("3a safe normalised difference", norm_diff(np.array([0.4, 0.0]), np.array([0.1, 0.0])),
      lambda v: np.isclose(v[0], 0.6) and np.isnan(v[1]),
      "(a - b) / (a + b), inside np.errstate(...), then replace non-finite values with np.nan.")
check("3b NDWI water mask", water,
      lambda v: v.shape == cls.shape and (v & (cls == 0)).sum() / (v | (cls == 0)).sum() > 0.9,
      "ndwi = norm_diff(refl[1], refl[3]); water = ndwi > 0.")

> **Where this goes:** Week 4 *Indices, and what each one is for*; Week 3 *Topographic
> correction* (why the cancellation in the demo is only approximately true on real slopes);
> Week 9 *Surface water*.

---
## 4 · Masks, logic and quality bits

### The idea

A **mask** is a True/False array with the image's shape: which pixels to keep. You build
masks with comparisons and combine them with `&` (and), `|` (or) and `~` (not).

**Always put comparisons in brackets:** `(ndvi > 0.3) & (ndwi < 0)`. Without the brackets,
Python reads `ndvi > 0.3 & ndwi < 0` as `ndvi > (0.3 & ndwi) < 0` and fails with *"truth
value of an array is ambiguous"*. It is the most common error in the first lab.

**Quality bands pack many yes/no flags into one integer**, one flag per **bit**. Landsat
Collection 2 `QA_PIXEL`, for example:

| bit | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| meaning | fill | dilated cloud | cirrus | cloud | cloud shadow | snow | clear | water |

The value 8 = binary `1000` means that only bit 3, *cloud*, is set. To test bit $k$:
`(qa & (1 << k)) != 0`. The `1 << k` is 2 to the power $k$, a number with only bit $k$ set.
Sentinel-2's `QA60` works the same way (bit 10 opaque cloud, bit 11 cirrus).

Some quality bands are **categories** rather than bits. Sentinel-2's Scene Classification
Layer (`SCL`) gives each pixel a single class code: 3 cloud shadow, 4 vegetation, 5 not
vegetated, 6 water, 8 and 9 cloud, 10 thin cirrus, 11 snow. For those you use `np.isin`.

### Demonstration

In [ ]:
for v in [64, 8, 10, 24, 128 + 64]:
    bits = np.binary_repr(v, width=8)
    set_bits = [k for k in range(8) if v & (1 << k)]
    print(f"QA {v:4d} = {bits}  bits set: {set_bits}")

### Check yourself

A synthetic `QA_PIXEL` array `qa` and an `SCL` array `scl` for a 100 × 100 scene are below.

**4a.** Build `clear`: True where **none** of *dilated cloud* (bit 1), *cloud* (bit 3) or
*cloud shadow* (bit 4) is set.

**4b.** Build `keep`: True where the SCL class is vegetation, not-vegetated or water (4, 5 or 6).

In [ ]:
rng = np.random.default_rng(40)
blob = ndimage.gaussian_filter(rng.standard_normal((100, 100)), 5)
cloud = blob > 0.08
dilated = ndimage.binary_dilation(cloud, iterations=2) & ~cloud
shadow = np.roll(cloud, (6, -6), axis=(0, 1)) & ~cloud & ~dilated
qa = np.where(cloud | dilated | shadow, 0, 1 << 6).astype(np.uint16)       # 'clear' bit
u16 = lambda m: m.astype(np.uint16)
qa |= (u16(cloud) << 3) | (u16(dilated) << 1) | (u16(shadow) << 4)
qa |= u16(rng.random((100, 100)) < 0.1) << 7                              # some 'water' flags
scl = rng.choice([3, 4, 5, 6, 8, 9, 10], size=(100, 100), p=[.05, .4, .25, .1, .08, .07, .05])

fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
ax[0].imshow(qa, cmap="tab20"); ax[0].set_title("QA_PIXEL (raw integers)")
ax[1].imshow(scl, cmap="tab10"); ax[1].set_title("SCL (class codes)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.show()

In [ ]:
clear = ...
keep = ...

<details><summary>Show a model answer</summary>

```python
bad = (1 << 1) | (1 << 3) | (1 << 4)          # one number with bits 1, 3 and 4 set
clear = (qa & bad) == 0
keep = np.isin(scl, [4, 5, 6])
print(f"clear: {clear.mean():.1%} of pixels   keep: {keep.mean():.1%}")
```

</details>

In [ ]:
check("4a clear-sky mask from bits", clear,
      lambda v: np.array_equal(v, (((qa >> 1) | (qa >> 3) | (qa >> 4)) & 1) == 0),
      "For each bit k: (qa & (1 << k)) != 0 marks it set. A pixel is clear when none of bits 1, 3, 4 is set.")
check("4b SCL mask", keep, lambda v: np.array_equal(v, (scl >= 4) & (scl <= 6)),
      "np.isin(scl, [4, 5, 6]).")

Note that bit 7 (*water*) is set on some clear pixels. A test such as `qa == 64` for
"clear" would silently throw those away. Test bits, never whole values.

> **Where this goes:** Week 2 *Filter, mask, composite*; Appendix A self-check; every
> practical after that.

---
## 5 · Stacks, filters and reducers

### The idea

Over one season you get many images of the same place: a **time stack** with shape
(time, rows, cols). Most Earth Engine work follows three steps:

1. **Filter**: keep only the images you want (date range, cloud cover in the metadata).
2. **Map**: apply the same function to every image (scale it, mask its clouds, add an index).
3. **Reduce**: collapse the time axis to one value per pixel (median, mean, max, count).

In NumPy, a reducer is a function with `axis=0`. For example, `np.nanmedian(stack, axis=0)`
is the median through time for every pixel at once, ignoring masked (`NaN`) values.

**Why the median?** Unmasked cloud is an extreme value: very bright in reflectance, and
NDVI near 0. The mean is dragged towards it. The median simply ignores values in the tails.
Masking first and then taking the median is better still.

Always also compute the **count** of clear observations per pixel. A composite pixel built
from one observation is not as trustworthy as one built from ten, and a pixel built from
zero should be `NaN`, not filled with a guess.

### Demonstration: twelve cloudy dates of NDVI

In [ ]:
rng = np.random.default_rng(50)
f = ndimage.gaussian_filter(rng.standard_normal((60, 60)), 4)
true_ndvi = 0.2 + 0.5 * (f - f.min()) / (f.max() - f.min())       # fields from 0.2 to 0.7

scenes, masks, meta = [], [], []
for i in range(12):
    cloud = ndimage.gaussian_filter(rng.standard_normal((60, 60)), 5) > rng.uniform(-0.1, 0.25)
    img = true_ndvi + rng.normal(0, 0.02, (60, 60))
    img[cloud] = rng.normal(0.02, 0.03, cloud.sum())                 # cloud: NDVI near 0
    scenes.append(img); masks.append(cloud)
    meta.append({"date": pd.Timestamp("2025-03-01") + pd.Timedelta(days=5 * i),
                 "cloud_cover": round(100 * cloud.mean(), 1)})
meta = pd.DataFrame(meta)
stack = np.stack(scenes)                                  # (12, 60, 60)
print(meta.to_string(index=False))

mean_c, median_c = stack.mean(axis=0), np.median(stack, axis=0)
for name, comp in [("mean composite", mean_c), ("median composite", median_c)]:
    print(f"{name:17s} mean abs error vs truth: {np.abs(comp - true_ndvi).mean():.3f}")

The median is about ten times closer to the truth than the mean. Masking the clouds before
taking the median does better still, and it also tells you which pixels were never seen
clearly at all. Now you do that properly.

### Check yourself

**5a. Filter, map, reduce.** Keep only the scenes with `cloud_cover < 60`. Set cloudy
pixels in each kept scene to `NaN` using `masks`, and take the per-pixel `np.nanmedian`
through time. Store the result as `composite`.

**5b.** Compute `n_clear`, the number of clear observations per pixel among the kept
scenes, and `n_empty`, the number of pixels with **none**.

In [ ]:
composite = ...
n_clear = ...
n_empty = ...

<details><summary>Show a model answer</summary>

```python
kept = meta.index[meta.cloud_cover < 60]                              # filter
masked = [np.where(masks[i], np.nan, scenes[i]) for i in kept]        # map
masked = np.stack(masked)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                                   # all-NaN pixels warn
    composite = np.nanmedian(masked, axis=0)                          # reduce
n_clear = np.sum(~np.isnan(masked), axis=0)
n_empty = int(np.sum(n_clear == 0))
print(f"kept {len(kept)} of 12 scenes; empty pixels: {n_empty}; "
      f"mean abs error: {np.nanmean(np.abs(composite - true_ndvi)):.3f}")
```

</details>

In [ ]:
_kept = meta.index[meta.cloud_cover < 60]
_n = np.sum([~masks[i] for i in _kept], axis=0)
check("5a masked median composite", composite,
      lambda v: v.shape == (60, 60) and np.nanmean(np.abs(v - true_ndvi)) < 0.03
                and np.array_equal(np.isnan(v), _n == 0),
      "Filter on meta.cloud_cover, np.where(mask, np.nan, scene) for each kept scene, then np.nanmedian(..., axis=0).")
check("5b clear-observation count", n_clear, lambda v: np.array_equal(v, _n),
      "Count the non-NaN values through time: np.sum(~np.isnan(masked), axis=0).")
check("5b empty pixels", n_empty, lambda v: v == int(np.sum(_n == 0)),
      "How many pixels have n_clear == 0?")

> **Where this goes:** Week 2 *Filter, mask, composite* and *Reducers*. In Earth Engine
> this whole exercise is about six lines, and the Earth Engine name for each step is in the
> table at the end.

---
## 6 · Time series, seasons and the Vegetation Condition Index

### The idea

Most of Kenya has **two rainy seasons**: the *long rains* (March–May) and the *short
rains* (October–December). Vegetation greens up a few weeks after each, so an NDVI time
series for a rangeland pixel has two humps a year.

A single NDVI value says little on its own: 0.3 is excellent in Marsabit in February
and a disaster in Kiambu in May. What matters is how a value compares with **the same
time of year in other years**, its **climatology**. Two ways to express that:

* **Anomaly:** value minus the long-term mean for that month (or a z-score).
* **Vegetation Condition Index**, used by Kenya's NDMA drought early-warning system:

$$ \text{VCI} = 100 \times \frac{\text{NDVI} - \text{NDVI}_{\min}}{\text{NDVI}_{\max} - \text{NDVI}_{\min}} $$

  where min and max are over **all years, for that same month**. So 0 means "the worst this
  month has been" and 100 means "the best". NDMA usually smooths it over 3 months
  (VCI3M) and reads it in bands: **<10 extreme**, **10–20 severe**, **20–35 moderate**
  drought, **35–50 normal**, **>50 above normal**.

Raw series are noisy (residual cloud, haze). **Smoothing**, e.g. a moving average,
trades noise for a slight blurring of timing.

### Demonstration: 20 years of monthly NDVI for one synthetic rangeland pixel

In [ ]:
rng = np.random.default_rng(60)
years = np.arange(2004, 2024)
months = np.arange(12)
long_rains = np.exp(-0.5 * ((months - 4) / 1.0) ** 2)      # peak in May
short_rains = np.exp(-0.5 * ((months - 10.5) / 1.0) ** 2)  # peak Nov-Dec
mam = rng.uniform(0.6, 1.2, len(years))                     # strength of each long-rains season
ond = rng.uniform(0.6, 1.2, len(years))                     # strength of each short-rains season
# planted failures, mirroring the droughts of 2010-11, 2016-17 and 2020-22
for y, s in [(2011, 0.25), (2017, 0.3), (2022, 0.15)]: mam[years == y] = s
for y, s in [(2010, 0.35), (2016, 0.3), (2021, 0.0)]:   ond[years == y] = s
ndvi = 0.18 + 0.22 * (mam[:, None] * long_rains + ond[:, None] * short_rains) \
       + rng.normal(0, 0.01, (len(years), 12))              # shape (20 years, 12 months)

series = pd.Series(ndvi.ravel(), index=pd.date_range("2004-01", periods=240, freq="MS"))
fig, ax = plt.subplots(figsize=(10, 3.4))
series.plot(ax=ax, lw=0.8, color="grey", label="monthly NDVI")
series.rolling(3, center=True).mean().plot(ax=ax, color="#B8443A", label="3-month mean")
ax.set(ylabel="NDVI", title="Two seasons a year, and some that failed"); ax.legend(); plt.show()

### Check yourself

**6a.** Compute `vci`, an array of the same shape as `ndvi` (years × months), using min and
max **per calendar month across years** (that is, along axis 0).

**6b.** For each year, average the VCI over **October–December** (month indices 9, 10, 11).
Which year had the worst short-rains season? Store it as `worst_ond` (an integer year).

In [ ]:
vci = ...
worst_ond = ...

<details><summary>Show a model answer</summary>

```python
lo, hi = ndvi.min(axis=0), ndvi.max(axis=0)          # one value per calendar month
vci = 100 * (ndvi - lo) / (hi - lo)
ond_vci = vci[:, 9:12].mean(axis=1)
worst_ond = int(years[ond_vci.argmin()])

bands = pd.cut(ond_vci, [-1, 10, 20, 35, 50, 101],
               labels=["extreme", "severe", "moderate", "normal", "above normal"])
print(pd.DataFrame({"OND VCI": ond_vci.round(0), "NDMA band": bands}, index=years))
print("worst:", worst_ond)
```

</details>

In [ ]:
check("6a VCI", vci,
      lambda v: np.shape(v) == ndvi.shape and np.allclose(v, 100 * (ndvi - ndvi.min(0)) / (ndvi.max(0) - ndvi.min(0))),
      "ndvi.min(axis=0) and ndvi.max(axis=0) give one value per month; broadcasting does the rest.")
check("6b worst short-rains season", worst_ond, lambda v: v == 2021,
      "vci[:, 9:12].mean(axis=1), then the year at argmin.")

Look at the table printed by the model answer. 2021 scores exactly 0. That is not because
the season was "zero percent" of anything, but because 2021 *is* the minimum of this record.
VCI is **relative to the record you have**: add one worse year, or drop the first ten years,
and every value in the table changes. Week 4 returns to this.

> **Where this goes:** Week 4 *Phenology* and *From index to decision: the Vegetation
> Condition Index*; Week 3 *Smoothing a time series*; Week 7 *Time-series algorithms*.

---
## 7 · Classification and accuracy

### The idea

**Supervised classification** learns a rule from labelled examples (training pixels):
*features* go in (band values, indices) and a *class* comes out. The rule is then applied to
every pixel.

To judge the map, compare it with **reference** labels the classifier did not see, in a
**confusion matrix**. In this unit's convention, **rows are the map and columns are the
reference**:

* **Overall accuracy (OA)** = the diagonal ÷ the total. It can look excellent while a small
  class is terrible.
* **User's accuracy** for class k = diagonal ÷ **row** total. *"If the map says forest, how
  often is it really forest?"* (1 − commission error).
* **Producer's accuracy** for class k = diagonal ÷ **column** total. *"Of the real forest,
  how much did the map find?"* (1 − omission error).

**The spatial trap again.** Pixels inside the same field are near-copies of each other.
If training and test pixels are drawn at random from the same fields, the test is almost
an exam you have seen the answers to. Split **by field** (by polygon) instead.

### Demonstration

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(70)
names = ["water", "cropland", "grassland", "bare"]
means = np.array([[.05, .04, .02, .01], [.08, .06, .35, .22], [.09, .09, .28, .26], [.14, .19, .26, .32]])
rows = []
for k, mu in enumerate(means):                       # 30 fields per class, 40 pixels per field
    for f in range(30):
        field_mu = mu + rng.normal(0, 0.035, 4)      # every field is a bit different
        px = field_mu + rng.normal(0, 0.008, (40, 4))
        rows += [(k, 100 * k + f, *p) for p in px]
px = pd.DataFrame(rows, columns=["cls", "field", "B3", "B4", "B8", "B11"])
X, y = px[["B3", "B4", "B8", "B11"]].values, px.cls.values

def oa_for(test):
    rf = RandomForestClassifier(100, random_state=0).fit(X[~test], y[~test])
    return (rf.predict(X[test]) == y[test]).mean(), rf

test_random = rng.random(len(px)) < 0.3
test_fields = px.field.isin(rng.choice(px.field.unique(), 36, replace=False)).values
oa_r, _ = oa_for(test_random)
oa_f, rf = oa_for(test_fields)
print(f"OA, random pixel split : {oa_r:.3f}")
print(f"OA, split by field     : {oa_f:.3f}   <- the honest one")
ref, mapped = y[test_fields], rf.predict(X[test_fields])

### Check yourself

From `ref` (reference labels) and `mapped` (the classifier's labels, from the field split)
for the four classes in `names`:

**7a.** Build the confusion matrix `cm` (4 × 4, **rows = map, columns = reference**) with
NumPy only. Then compute `oa`, and arrays `users` and `producers` with one value per class.

**7b.** Which class has the **lowest producer's accuracy**? Store its name in `worst_pa`.

In [ ]:
cm = ...
oa = ...
users = ...
producers = ...
worst_pa = ...

<details><summary>Show a model answer</summary>

```python
cm = np.zeros((4, 4), int)
np.add.at(cm, (mapped, ref), 1)          # row = map label, column = reference label
oa = np.trace(cm) / cm.sum()
users = np.diag(cm) / cm.sum(axis=1)
producers = np.diag(cm) / cm.sum(axis=0)
worst_pa = names[producers.argmin()]
print(pd.DataFrame(cm, index=[f"map {n}" for n in names], columns=[f"ref {n}" for n in names]))
print(f"OA {oa:.3f}\nuser's     {users.round(3)}\nproducer's {producers.round(3)}\nworst PA: {worst_pa}")
```

</details>

In [ ]:
_cm = pd.crosstab(pd.Categorical(mapped, range(4)), pd.Categorical(ref, range(4)), dropna=False).values
check("7a confusion matrix", cm, lambda v: np.array_equal(v, _cm),
      "Rows = map (mapped), columns = reference (ref). np.add.at(cm, (mapped, ref), 1) fills it.")
check("7a overall accuracy", oa, lambda v: np.isclose(v, np.trace(_cm) / _cm.sum()), "Diagonal sum / total.")
check("7a user's and producer's", (users, producers) if users is not ... else ...,
      lambda v: np.allclose(v[0], np.diag(_cm) / _cm.sum(1)) and np.allclose(v[1], np.diag(_cm) / _cm.sum(0)),
      "User's divides the diagonal by ROW totals (axis=1); producer's by COLUMN totals (axis=0).")
check("7b worst producer's accuracy", worst_pa,
      lambda v: v == names[int(np.argmin(np.diag(_cm) / _cm.sum(0)))],
      "names[producers.argmin()].")

One more idea to take into Week 6: **counting mapped pixels is not an area estimate.**
If grassland is often mapped as cropland, the cropland pixel count is inflated and the
errors do not cancel. Week 6 shows how to correct area using the confusion matrix and a
proper sample.

> **Where this goes:** Week 5 *Supervised classification* (especially *Overfitting, and how
> to see it early*); Week 6 *Accuracy assessment done properly*, the centre of the unit.

---
## 8 · Logarithms and decibels, for radar

### The idea

Radar (Sentinel-1) sends its own microwave pulse and measures the **backscatter**
$\sigma^0$ that returns. This is a ratio of powers, and it spans a huge range: calm water
returns about 0.003, a town can return more than 1. So it is reported in **decibels**:

$$ \sigma^0_{\text{dB}} = 10\,\log_{10}\sigma^0 \qquad\qquad \sigma^0 = 10^{\,\sigma^0_{\text{dB}}/10} $$

| dB | 0 | −3 | −10 | −20 | −25 |
|---|---|---|---|---|---|
| linear | 1 | ≈ 0.5 | 0.1 | 0.01 | ≈ 0.003 |

So +3 dB is roughly twice the power, and every −10 dB is another factor of ten.

Radar images are covered in **speckle**, a grainy noise that *multiplies* the signal
(a single look is exponentially distributed around the true value). Two consequences:

* **Filter and average in linear units, then convert to dB.** The log of an average is not
  the average of the logs. For single-look speckle, averaging in dB comes out about
  **2.5 dB too low**, every time.
* Earth Engine's `COPERNICUS/S1_GRD` collection is **already in dB**. Convert to linear
  before you average or filter it.

### Demonstration

In [ ]:
rng = np.random.default_rng(80)
truth_water = np.zeros((120, 120), bool)
truth_water[20:70, 15:90] = True                                   # a flooded area
sigma0 = np.where(truth_water, 10 ** (-23 / 10), 10 ** (-12 / 10))  # water -23 dB, grass -12 dB
intensity = sigma0 * rng.exponential(1.0, sigma0.shape)             # single-look speckle
img_db = 10 * np.log10(intensity)

g = ~truth_water
print(f"grass, true value               : {-12:.2f} dB")
print(f"grass, mean of dB values        : {img_db[g].mean():.2f} dB   <- biased low")
print(f"grass, dB of mean linear values : {10*np.log10(intensity[g].mean()):.2f} dB")

plt.imshow(img_db, cmap="gray", vmin=-35, vmax=0); plt.colorbar(label="dB")
plt.title("Single-look backscatter (dB): the flood is there, under speckle"); plt.axis("off"); plt.show()

### Check yourself

**8a.** Write `to_db(x)` and `from_db(d)`.

**8b.** Five neighbouring pixels read `[-10, -13, -16, -11, -20]` dB. Their correct mean,
in dB, is `mean_db`.

**8c.** Map the flood: smooth `intensity` with a 5 × 5 moving average **in linear units**
(`ndimage.uniform_filter(intensity, 5)`), convert to dB, and call pixels below −18 dB water.
Store the mask in `flood`. The check wants more than 95% of pixels right. Try thresholding
`img_db` directly without smoothing first, and compare.

In [ ]:
def to_db(x):
    return ...

def from_db(d):
    return ...

mean_db = ...
flood = ...

<details><summary>Show a model answer</summary>

```python
def to_db(x):
    return 10 * np.log10(x)

def from_db(d):
    return 10 ** (np.asarray(d, float) / 10)

mean_db = to_db(from_db([-10, -13, -16, -11, -20]).mean())
flood = to_db(ndimage.uniform_filter(intensity, 5)) < -18

print(f"mean {mean_db:.2f} dB (naive dB average would be {np.mean([-10, -13, -16, -11, -20]):.1f})")
print(f"accuracy, filtered: {(flood == truth_water).mean():.3f}   "
      f"unfiltered: {((img_db < -18) == truth_water).mean():.3f}")
```

</details>

In [ ]:
check("8a to_db", to_db(np.array([1, 0.1, 0.01])),
      lambda v: np.allclose(v, [0, -10, -20]), "10 * np.log10(x).")
check("8a from_db", from_db(np.array([0, -10, -20])),
      lambda v: np.allclose(v, [1, 0.1, 0.01]), "10 ** (d / 10).")
check("8b mean of dB values", mean_db,
      lambda v: np.isclose(v, 10 * np.log10(np.mean(10 ** (np.array([-10, -13, -16, -11, -20]) / 10)))),
      "Convert each value to linear, average, convert back. The answer is not -14.")
check("8c flood map", flood,
      lambda v: v.shape == truth_water.shape and (v == truth_water).mean() > 0.95,
      "Smooth the LINEAR intensity with ndimage.uniform_filter(intensity, 5), then to dB, then < -18.")

> **Where this goes:** Week 8 *What backscatter measures* and *Speckle*; Week 9 *Surface water*.

---
## From NumPy to Earth Engine

Everything above has an Earth Engine equivalent. The ideas are identical, only the names
change. Keep this table open during Weeks 1–3.

| What you did here (NumPy) | Earth Engine (Code Editor, JavaScript) |
|---|---|
| `(dn - 1000) / 10000` | `image.multiply(0.0001)` on `S2_SR_HARMONIZED` (offset already removed) |
| `norm_diff(nir, red)` | `image.normalizedDifference(['B8', 'B4'])` |
| `refl[:, mask]`, `np.where(mask, x, np.nan)` | `image.updateMask(mask)` |
| `(qa & (1 << 10)) == 0` | `qa.bitwiseAnd(1 << 10).eq(0)` |
| `np.isin(scl, [4, 5, 6])` | `scl.eq(4).or(scl.eq(5)).or(scl.eq(6))` |
| filter scenes by `meta.cloud_cover` | `collection.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))` |
| filter by date | `collection.filterDate('2025-03-01', '2025-05-31')` |
| `[f(s) for s in scenes]` | `collection.map(f)` |
| `np.nanmedian(stack, axis=0)` | `collection.median()`: masked pixels are skipped automatically |
| `np.sum(~np.isnan(stack), axis=0)` | `collection.count()` |
| `np.sum(mask) * 100` (m²) | `mask.multiply(ee.Image.pixelArea()).reduceRegion({reducer: ee.Reducer.sum(), geometry: aoi, scale: 10})` |
| confusion matrix from two label arrays | `validated.errorMatrix('class', 'classification')` |
| `10 * np.log10(x)` | `image.log10().multiply(10)` |
| `10 ** (d / 10)` | `ee.Image(10).pow(image.divide(10))` |

One difference will feel strange at first. In Earth Engine, your script *describes* the
computation, and Google's servers run it only when you ask for a result (`print`,
`Map.addLayer`, `Export`). Week 2, *The one concept that matters*, is about exactly that.

---
## Before Week 1: can you say yes to each of these?

- [ ] I can sketch the signatures of vegetation, water and soil, and point to the red edge.
- [ ] I check scale factors and offsets before any maths, and I never subtract in `uint16`.
- [ ] I can explain why ratios cancel illumination but not haze.
- [ ] I can build masks with brackets, `&`, `|` and `~`, and test individual QA bits.
- [ ] I can filter, map and reduce a stack, and I report a clear-observation count with a composite.
- [ ] I can compute VCI and say what a value of 15 means for a pastoralist.
- [ ] I can build a confusion matrix and tell user's accuracy from producer's accuracy.
- [ ] I convert dB to linear before averaging or filtering radar.

Any box you cannot tick: go back to that section. Then do the reader's *Earth Engine
self-check* (Appendix A) once your account is active. See you in Week 1.